### Get Reactome GMT


In [1]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"
ROOT0_DATA = ROOT0 / "data"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import pdwritecsv, pdreadcsv, create_dir, write_txt
from libs.enricher_lib import enricheR
from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [2]:
root0 = Path('/home/flavio/uv/perturb_agent')
root0_data = root0 / 'data'
root_colab = root0_data / 'colab'

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-ACC'

disease = PSI_ID

root_project = create_dir(root0_data, PROG_ID)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=root0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={ptw_min_num_of_degs_cut}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-ACC/config/all_lfc_cutoffs_TCGA-ACC.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [3]:
root_owl = root_colab / 'owl'
root_reactome = root_colab / 'reactome'

In [4]:

from libs.reactome_lib import Reactome


rea = Reactome(root_owl=root_owl, root_reactome=root_reactome)

In [5]:
fname = rea.fname_reactome_gmt
dfrea = pdreadcsv(fname, rea.root_reactome)
dfrea.head(3)

,pathway_id,pathway,genes,n
0,R-HSA-164843,2-LTR Circle Formation,"['REV', 'XRCC5', 'XRCC6', 'XRCC4', 'PSIP1', 'LIG4', 'HMGA1', 'GAG', 'GAG-POL...",13
1,R-HSA-1971475,A Tetrasaccharide Linker Sequence Is Required for GAG Synthesis,"['B3GAT3', 'B3GAT2', 'B3GAT1', 'BGN', 'B3GALT6', 'DCN', 'VCAN', 'BCAN', 'HSP...",26
2,R-HSA-5619084,ABC Transporter Disorders,"['ABCG8', 'ABCC2', 'KCNJ11', 'ABCC8', 'ABCC9', 'ABCC6', 'PSMC6', 'ABCA12', '...",65


In [6]:
if rea.df_gmt.empty:
    df_gmt = rea.open_reactome_gmt(verbose=True)

Table opened ((2119, 4)) at '/home/flavio/uv/perturb_agent/data/colab/reactome/reactome_pathways_human_gmt.tsv'


In [7]:
pathway_id = 'R-HSA-1971475'
genes = rea.df_gmt.loc[rea.df_gmt['pathway_id'] == pathway_id].iloc[0]['genes']
if isinstance(genes, str):
    genes = eval(genes)

type(genes), len(genes), ";".join(genes)

(list,
 26,
 'B3GAT3;B3GAT2;B3GAT1;BGN;B3GALT6;DCN;VCAN;BCAN;HSPG2;SDC1;SDC2;CSPG5;CSPG4;B4GALT7;AGRN;NCAN;SDC3;XYLT1;SDC4;XYLT2;GPC2;GPC1;GPC4;GPC3;GPC6;GPC5')